# Notebook 03: Regression Analysis

Log-log regression of complexity metrics against runtime.

- **Development set** (n=27): tandem, fork-join, feedback models
- **Out-of-sample set** (n=9): hybrid models (3x3 factorial: N in {2,3,4}, M in {2,3,4})
- R² computed in log-space using training set mean as baseline
- Diebold-Mariano tests for comparing forecast accuracy

In [ ]:
import sys
import os
import json
from pathlib import Path

import numpy as np
from scipy import stats

sys.path.insert(0, os.path.abspath('../src'))
from utils import (load_metrics, load_runtime, merge_metrics_runtime,
                    fit_loglog, predict_loglog, compute_r2, compute_r2_logspace,
                    compute_mape, compute_rmse, diebold_mariano, spearman_rank)

DATA_DIR = Path('../data')
OUTPUT_DIR = Path('../output')
OUTPUT_DIR.mkdir(exist_ok=True)

print('Setup complete.')

## 1. Load and merge data

In [ ]:
# Load metrics — prefer notebook 01 output, fall back to pre-computed
computed_path = OUTPUT_DIR / 'computed_metrics.json'
if computed_path.exists():
    raw = load_metrics(computed_path)
    # computed_metrics.json is {"models": [...]}, with "name" instead of "model_name"
    if isinstance(raw, dict) and 'models' in raw:
        metrics = raw['models']
        for m in metrics:
            if 'model_name' not in m and 'name' in m:
                m['model_name'] = m['name']
    else:
        metrics = raw
    print(f'Loaded metrics from notebook 01 output ({len(metrics)} models)')
else:
    metrics = load_metrics(DATA_DIR / 'results' / 'all_metrics.json')
    print(f'Loaded pre-computed metrics ({len(metrics)} models)')

# Load runtime data
runtime = load_runtime(DATA_DIR / 'results' / 'hybrid_runtime.json')
merged = merge_metrics_runtime(metrics, runtime)

# Split into dev and OOS sets
dev_set = [m for m in merged if m['topology'] in ('tandem', 'fork_join', 'feedback')]

# OOS: 3x3 factorial design (N in {2,3,4}, M in {2,3,4}) to match the paper
oos_factorial = {f'hybrid_{n}_{m}' for n in (2, 3, 4) for m in (2, 3, 4)}
oos_set = [m for m in merged if m['model_name'] in oos_factorial]

print(f'Total merged: {len(merged)} models')
print(f'Development set: {len(dev_set)} models')
print(f'OOS (hybrid 3x3 factorial) set: {len(oos_set)} models')
print(f'OOS models: {sorted(m["model_name"] for m in oos_set)}')

# Extract arrays for dev set
dev_names = [m['model_name'] for m in dev_set]
dev_smc = np.array([m['smc'] for m in dev_set], dtype=float)
dev_cc = np.array([m['cyclomatic_number'] for m in dev_set], dtype=float)
dev_loc = np.array([m['loc'] for m in dev_set], dtype=float)
dev_kc = np.array([m['kc'] for m in dev_set], dtype=float)
dev_rt = np.array([m['wall_time'] for m in dev_set], dtype=float)

# Extract arrays for OOS set
oos_names = [m['model_name'] for m in oos_set]
oos_smc = np.array([m['smc'] for m in oos_set], dtype=float)
oos_cc = np.array([m['cyclomatic_number'] for m in oos_set], dtype=float)
oos_loc = np.array([m['loc'] for m in oos_set], dtype=float)
oos_kc = np.array([m['kc'] for m in oos_set], dtype=float)
oos_rt = np.array([m['wall_time'] for m in oos_set], dtype=float)

## 2. Log-log regression on development set (Table 2)

In [ ]:
# Fit log-log models: log(runtime) = intercept + slope * log(metric)
metrics_dict = {
    'SMC': dev_smc,
    'CC': dev_cc,
    'LOC': dev_loc,
    'KC': dev_kc,
}

fit_results = {}
print('Table 2: Log-Log Regression Results (Development Set, n={})'.format(len(dev_set)))
print(f"{'Metric':<8s} {'Slope':>8s} {'Intercept':>10s} {'R²':>8s} {'p-value':>12s} {'Std Err':>10s}")
print('-' * 62)

for name, x in metrics_dict.items():
    result = fit_loglog(x, dev_rt)
    fit_results[name] = result
    print(f"{name:<8s} {result['slope']:>8.4f} {result['intercept']:>10.4f} "
          f"{result['r2']:>8.4f} {result['p_value']:>12.2e} {result['std_err']:>10.4f}")

## 3. Out-of-sample prediction on hybrid set (Table 3)

Log-space R² with training set mean as baseline, matching the paper methodology.

In [ ]:
oos_metrics_dict = {
    'SMC': oos_smc,
    'CC': oos_cc,
    'LOC': oos_loc,
    'KC': oos_kc,
}

oos_predictions = {}
oos_errors = {}

print('Table 3: Out-of-Sample Prediction (Hybrid 3x3 Factorial, n={})'.format(len(oos_set)))
print(f"{'Metric':<8s} {'R²(log)':>8s} {'MAPE(%)':>10s} {'RMSE':>8s}")
print('-' * 38)

for name, x in oos_metrics_dict.items():
    fr = fit_results[name]
    pred = predict_loglog(x, fr['slope'], fr['intercept'])
    oos_predictions[name] = pred
    oos_errors[name] = oos_rt - pred
    
    r2 = compute_r2_logspace(oos_rt, pred, dev_rt)
    mape = compute_mape(oos_rt, pred)
    rmse = compute_rmse(oos_rt, pred)
    print(f"{name:<8s} {r2:>8.4f} {mape:>10.2f} {rmse:>8.4f}")

## 4. Diebold-Mariano tests (Table 4)

In [ ]:
# Pairwise Diebold-Mariano tests comparing OOS forecast accuracy
# H0: equal predictive accuracy
# Negative DM statistic => first model is better

metric_names = list(oos_errors.keys())

print('Table 4: Diebold-Mariano Tests (pairwise comparisons)')
print(f"{'Comparison':<20s} {'DM stat':>10s} {'p-value':>10s} {'Better':>10s}")
print('-' * 54)

for i in range(len(metric_names)):
    for j in range(i + 1, len(metric_names)):
        name_i = metric_names[i]
        name_j = metric_names[j]
        dm, pval = diebold_mariano(oos_errors[name_i], oos_errors[name_j])
        better = name_i if dm < 0 else name_j
        sig = '*' if pval < 0.05 else ''
        print(f"{name_i + ' vs ' + name_j:<20s} {dm:>10.4f} {pval:>10.4f} {better + sig:>10s}")

## 5. Spearman rank correlations

In [ ]:
# Spearman rank correlations between metrics and runtime (dev set)
print('Spearman Rank Correlations (Development Set)')
print(f"{'Metric':<8s} {'rho':>8s} {'p-value':>12s}")
print('-' * 32)

for name, x in metrics_dict.items():
    rho, pval = spearman_rank(x, dev_rt)
    print(f"{name:<8s} {rho:>8.4f} {pval:>12.2e}")

# Cross-correlations between metrics
print('\nPairwise Metric Correlations')
print(f"{'Pair':<20s} {'rho':>8s} {'p-value':>12s}")
print('-' * 44)
for i in range(len(metric_names)):
    for j in range(i + 1, len(metric_names)):
        name_i = metric_names[i]
        name_j = metric_names[j]
        x_i = metrics_dict.get(name_i, oos_metrics_dict.get(name_i))
        x_j = metrics_dict.get(name_j, oos_metrics_dict.get(name_j))
        if x_i is not None and x_j is not None:
            rho, pval = spearman_rank(x_i, x_j)
            print(f"{name_i + ' vs ' + name_j:<20s} {rho:>8.4f} {pval:>12.2e}")

## 6. Summary statistics

In [ ]:
print('Summary Statistics')
print('=' * 70)

for label, data, mdict in [('Development Set', dev_set, metrics_dict), 
                            ('OOS Set', oos_set, oos_metrics_dict)]:
    print(f'\n{label} (n={len(data)})')
    rt = np.array([m['wall_time'] for m in data])
    print(f"  Runtime: min={rt.min():.3f}s, max={rt.max():.3f}s, "
          f"mean={rt.mean():.3f}s, median={np.median(rt):.3f}s")
    for name, x in mdict.items():
        print(f"  {name}: min={x.min():.2f}, max={x.max():.2f}, "
              f"mean={x.mean():.2f}, median={np.median(x):.2f}")

# Save regression results
results_out = {
    'fit_results': {k: {kk: float(vv) for kk, vv in v.items()} for k, v in fit_results.items()},
    'n_dev': len(dev_set),
    'n_oos': len(oos_set),
}
with open(OUTPUT_DIR / 'regression_results.json', 'w') as f:
    json.dump(results_out, f, indent=2)
print(f'\nSaved regression results to output/regression_results.json')